Dataset: https://www.kaggle.com/datasets/belbino/global-fuel-prices-20202026
Partners: George Rodriguez and Alexis Velasquez

In [ ]:
# Import necessary libraries
import xgboost as xgb  # XGBoost library
from sklearn.model_selection import train_test_split  # For splitting data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Evaluation metrics
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving global_fuel_prices_2020_2026.csv to global_fuel_prices_2020_2026.csv


In [ ]:
# ============================================================================
# STEP 1: LOAD AND PREPARE THE DATA
# ============================================================================

# Load dataset
df = pd.read_csv("global_fuel_prices_2020_2026.csv")

print("Original shape:", df.shape)
print(df.head())

Original shape: (27468, 10)
         date        country         region income_level subsidy_level  \
0  2020-01-06  United States  North America         High           Low   
1  2020-01-13  United States  North America         High           Low   
2  2020-01-20  United States  North America         High           Low   
3  2020-01-27  United States  North America         High           Low   
4  2020-02-03  United States  North America         High           Low   

   petrol_usd_liter  diesel_usd_liter  lpg_usd_liter  brent_crude_usd  \
0             1.465             1.289          1.093            65.75   
1             1.435             1.319          1.077            65.54   
2             1.446             1.329          1.080            66.51   
3             1.488             1.359          1.125            68.79   
4             1.457             1.341          1.111            68.44   

   tax_percentage  
0            59.6  
1            27.6  
2            62.3  
3       

In [ ]:
# Drop columns we are not using
df = df.drop(columns=["income_level", "subsidy_level"])

In [ ]:
# Drop missing values
df = df.dropna()
print("Shape after cleaning:", df.shape)

Shape after cleaning: (27468, 8)


In [ ]:
# Convert date into a usable numeric format
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

In [ ]:
# Drop original date column after extracting useful parts
df = df.drop(columns=["date"])

In [ ]:
# Create fuel price categories: 3 classes (Low, Medium, High)
df["price_class"] = pd.qcut(df["petrol_usd_liter"], q=3, labels=[0, 1, 2])

In [ ]:
# One-hot encode categorical columns
df = pd.get_dummies(df, columns=["country", "region"], drop_first=True)

In [ ]:
# Extract features (X) and target labels (y)
X = df.drop(columns=["petrol_usd_liter", "price_class"])
y = df["price_class"].astype(int)

In [ ]:
# Create DataFrame for better visualization
df_display = X.copy()
df_display["price_class"] = y

print("=" * 70)
print("ORIGINAL DATASET INFO")
print("=" * 70)
print(f"Total samples: {len(df_display)}")
print(f"\nFirst 5 rows:")
print(df_display.head())

print(f"\nClass distribution:")
print(df_display["price_class"].value_counts().sort_index())

ORIGINAL DATASET INFO
Total samples: 27468

First 5 rows:
   diesel_usd_liter  lpg_usd_liter  brent_crude_usd  tax_percentage  year  \
0             1.289          1.093            65.75            59.6  2020   
1             1.319          1.077            65.54            27.6  2020   
2             1.329          1.080            66.51            62.3  2020   
3             1.359          1.125            68.79            44.8  2020   
4             1.341          1.111            68.44            51.4  2020   

   month  day  country_Angola  country_Argentina  country_Australia  ...  \
0      1    6           False              False              False  ...   
1      1   13           False              False              False  ...   
2      1   20           False              False              False  ...   
3      1   27           False              False              False  ...   
4      2    3           False              False              False  ...   

   country_Vietnam  co

In [ ]:
# ============================================================================
# STEP 2: MULTI-CLASS CLASSIFICATION SETUP
# Convert petrol prices into 3 classes:
# 0 = Low, 1 = Medium, 2 = High
# ============================================================================

print("\n" + "=" * 70)
print("AFTER CLASS CONVERSION")
print("=" * 70)
print(f"Class 0 (Low Price): {sum(y == 0)} samples")
print(f"Class 1 (Medium Price): {sum(y == 1)} samples")
print(f"Class 2 (High Price): {sum(y == 2)} samples")


AFTER CLASS CONVERSION
Class 0 (Low Price): 9158 samples
Class 1 (Medium Price): 9157 samples
Class 2 (High Price): 9153 samples


In [ ]:
# ============================================================================
# STEP 3: SPLIT DATA INTO TRAINING AND TESTING SETS
# ============================================================================

# Split: 70% training, 30% testing
# stratify ensures each class is proportionally represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,       # 30% for testing
    random_state=42,     # For reproducibility
    stratify=y           # Maintain class proportions
)

print(f"\nTraining set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")


Training set size: 19227
Testing set size: 8241


In [ ]:
# Check class distribution in splits
print(f"\nClass distribution in training set:")
for class_id in range(3):
    count = sum(y_train == class_id)
    print(f"  Class {class_id}: {count} samples")

print(f"\nClass distribution in testing set:")
for class_id in range(3):
    count = sum(y_test == class_id)
    print(f"  Class {class_id}: {count} samples")


Class distribution in training set:
  Class 0: 6410 samples
  Class 1: 6410 samples
  Class 2: 6407 samples

Class distribution in testing set:
  Class 0: 2748 samples
  Class 1: 2747 samples
  Class 2: 2746 samples


In [ ]:
# ============================================================================
# STEP 4: CREATE AND TRAIN THE XGBOOST CLASSIFIER
# ============================================================================

print("\n" + "=" * 70)
print("TRAINING XGBOOST MULTI-CLASS CLASSIFIER")
print("=" * 70)


TRAINING XGBOOST MULTI-CLASS CLASSIFIER


In [ ]:
# Initialize XGBoost Classifier for multi-class classification
model = xgb.XGBClassifier(
    max_depth=3,              # Maximum depth of trees (shallow = simpler)
    n_estimators=100,         # Number of boosting rounds
    learning_rate=0.1,        # Step size shrinkage
    objective='multi:softmax',  # Multi-class classification (returns class labels)
    num_class=3,              # Number of classes
    random_state=42,          # For reproducibility
    eval_metric='mlogloss'    # Multi-class log loss
)

print("Model parameters:")
print(f"  - Objective: {model.objective} (multi-class classification)")
print(f"  - Number of classes: {len(np.unique(y))}")
print(f"  - Max tree depth: {model.max_depth}")
print(f"  - Number of trees: {model.n_estimators}")
print(f"  - Learning rate: {model.learning_rate}")

Model parameters:
  - Objective: multi:softmax (multi-class classification)
  - Number of classes: 3
  - Max tree depth: 3
  - Number of trees: 100
  - Learning rate: 0.1


In [ ]:
# Train the model
print("\nTraining in progress...")
model.fit(X_train, y_train)
print("✓ Training complete!")


Training in progress...
✓ Training complete!


In [ ]:
# ============================================================================
# STEP 5: MAKE PREDICTIONS
# ============================================================================

# Predict on training set
y_train_pred = model.predict(X_train)

# Predict on test set
y_test_pred = model.predict(X_test)

# Get prediction probabilities
model_proba = xgb.XGBClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=3,
    random_state=42
)
model_proba.fit(X_train, y_train)
y_test_proba = model_proba.predict_proba(X_test)

In [ ]:
# ============================================================================
# STEP 6: EVALUATE THE MODEL
# ============================================================================

print("\n" + "=" * 70)
print("MODEL EVALUATION")
print("=" * 70)


MODEL EVALUATION


In [ ]:
# Accuracy scores
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy:  {test_accuracy:.4f}")

Training Accuracy: 0.9946
Testing Accuracy:  0.9930


In [ ]:
# Classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2748
           1       0.99      0.99      0.99      2747
           2       0.99      1.00      1.00      2746

    accuracy                           0.99      8241
   macro avg       0.99      0.99      0.99      8241
weighted avg       0.99      0.99      0.99      8241



In [ ]:
# Confusion matrix
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Confusion Matrix:
[[2732   16    0]
 [  19 2714   14]
 [   0    9 2737]]


In [ ]:
# Show probability predictions for first 5 test samples
print(f"\nFirst 5 probability predictions:")
print(y_test_proba[:5])


First 5 probability predictions:
[[3.8416696e-05 1.1938547e-04 9.9984217e-01]
 [9.9979502e-01 1.5544680e-04 4.9564005e-05]
 [1.9914618e-04 9.9960297e-01 1.9787891e-04]
 [2.5600134e-04 9.9948955e-01 2.5437248e-04]
 [2.2875423e-04 9.9954396e-01 2.2729876e-04]]


# Summary

This project uses a global dataset of fuel prices from 2020 to 2026 to explore how prices vary across countries and regions over time. The data is cleaned and simplified, then each petrol price is grouped into one of three categories: low, medium, or high. Using a machine learning method called Extreme Gradient Boosting (XGBoost), the program learns patterns from factors like location and date to predict which price category a given entry belongs to. The model is trained on most of the data and then tested on unseen data to check how well it performs. Key highlights include organizing real-world data into meaningful groups, applying an advanced predictive model, and evaluating its accuracy using clear metrics like overall correctness and comparison tables. Overall, the project demonstrates how machine learning can help identify patterns in global fuel pricing in a way that is easy to interpret.
